# Qwen3 Embedding GPU endpoint cho Research-assist

Notebook này **chạy inference**. Fine-tune bằng notebook **06** trước.
Chọn Qwen/Qwen3-Embedding-0.6B
hoặc Qwen/Qwen3-Embedding-4B, hoặc trỏ tới thư mục checkpoint Qwen3 đã huấn luyện.
Với T4, thử 0.6B trước; 4B có thể hết VRAM tùy độ dài chuỗi và GPU còn trống.
Không cộng VRAM của hai T4 vào một tiến trình.

Endpoint sử dụng `encode_query` / `encode_document`, khác prefix của E5.
Notebook mặc định không mở server/tunnel. Chỉ bật tunnel nếu nền tảng cho phép;
URL tạm chỉ sống trong phiên chạy. Không lưu key trong output notebook.

Nguồn: https://huggingface.co/Qwen/Qwen3-Embedding-4B

In [ ]:
import os, sys, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "sentence-transformers==5.1.2", "transformers==4.57.1",
    "fastapi>=0.115,<1", "uvicorn>=0.30,<1", "httpx>=0.27,<1",
    "huggingface-hub==0.36.0", "peft==0.17.1", "bitsandbytes==0.48.2", "accelerate==1.11.0"])

In [ ]:
"""Inference for the SWIFT Qwen3 adapter recipe; identical query/EOS policy to training."""

import hashlib
import json
from pathlib import Path


def adapter_identity(path, max_seq_length):
    root = Path(path)
    digest = hashlib.sha256(f"swift-qwen3-eos-v1:{max_seq_length}".encode())
    for name in ("research_model.json", "adapter_config.json", "adapter_model.safetensors"):
        digest.update(name.encode())
        with (root / name).open("rb") as stream:
            while block := stream.read(1024 * 1024):
                digest.update(block)
    return "research-qwen3-" + digest.hexdigest()[:32]


def read_adapter_manifest(path):
    root = Path(path)
    manifest = json.loads((root / "research_model.json").read_text(encoding="utf-8"))
    if (
        manifest.get("format") != "swift_qwen3_adapter_v1"
        or manifest.get("model_family") != "qwen3"
        or manifest.get("fine_tuned") is not True
        or manifest.get("optimizer_steps", 0) < 1
        or manifest.get("base_model")
        not in {"Qwen/Qwen3-Embedding-4B", "Qwen/Qwen3-Embedding-0.6B"}
        or len(manifest.get("base_revision", "")) != 40
        or not manifest.get("query_instruction")
        or not (root / "adapter_model.safetensors").is_file()
    ):
        raise ValueError("Invalid trained Qwen3 adapter manifest; base fallback is forbidden.")
    return manifest


def extract_adapter_archive(archive_path, destination):
    import zipfile

    root = Path(destination).resolve()
    if root.exists() and any(root.iterdir()):
        raise ValueError("Choose an empty adapter extraction directory.")
    with zipfile.ZipFile(archive_path) as archive:
        size = 0
        for item in archive.infolist():
            target = (root / item.filename.replace("\\", "/")).resolve()
            if not target.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Unsafe adapter archive entry.")
            size += item.file_size
            if size > 2 * 1024**3:
                raise ValueError("Adapter archive exceeds 2 GiB.")
        archive.extractall(root)
    read_adapter_manifest(root)
    return root


class QwenAdapterEncoder:
    """Load a pinned base plus optional adapter. Use base-only solely for baseline evaluation."""

    def __init__(self, base_path, instruction, max_seq_length=512, adapter_path=None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        self.max_seq_length, self.instruction = max_seq_length, instruction
        self.tokenizer = AutoTokenizer.from_pretrained(str(base_path), trust_remote_code=False)
        self.tokenizer.padding_side = "right"
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.suffix = self.tokenizer.convert_tokens_to_ids("<|endoftext|>")
        if self.suffix is None or self.suffix == self.tokenizer.unk_token_id:
            raise ValueError("Qwen tokenizer must contain the SWIFT endoftext suffix.")
        model = AutoModelForCausalLM.from_pretrained(
            str(base_path),
            trust_remote_code=False,
            local_files_only=True,
            torch_dtype=torch.float16,
            attn_implementation="sdpa",
            device_map={"": 0},
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            ),
        )
        if adapter_path:
            from peft import PeftModel

            model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)
        self.model = model.eval()

    def get_sentence_embedding_dimension(self):
        return int(self.model.config.hidden_size)

    def encode_query(self, texts, **kwargs):
        return self._encode(
            [f"Instruct: {self.instruction}\nQuery:{text}" for text in texts], **kwargs
        )

    def encode_document(self, texts, **kwargs):
        return self._encode(texts, **kwargs)

    def _encode(self, texts, batch_size=2, **kwargs):
        import numpy as np
        import torch
        import torch.nn.functional as functional

        result = []
        with torch.inference_mode():
            for start in range(0, len(texts), batch_size):
                ids = self.tokenizer(
                    texts[start : start + batch_size],
                    add_special_tokens=False,
                    truncation=True,
                    max_length=self.max_seq_length - 1,
                )["input_ids"]
                batch = self.tokenizer.pad(
                    {"input_ids": [row + [self.suffix] for row in ids]},
                    padding=True,
                    return_tensors="pt",
                ).to(self.model.device)
                base = (
                    self.model.get_base_model()
                    if hasattr(self.model, "peft_config")
                    else self.model
                )
                # Use the transformer with adapters; avoid the vocabulary-sized LM head.
                hidden = base.model(**batch, use_cache=False, return_dict=True).last_hidden_state
                last = batch["attention_mask"].sum(dim=1) - 1
                vectors = hidden[torch.arange(len(last), device=hidden.device), last].float()
                result.append(functional.normalize(vectors, dim=1).cpu().numpy())
        return (
            np.concatenate(result)
            if result
            else np.empty((0, self.get_sentence_embedding_dimension()))
        )


def load_swift_qwen_adapter(path, max_seq_length=512):
    from huggingface_hub import snapshot_download

    manifest = read_adapter_manifest(path)
    base = snapshot_download(manifest["base_model"], revision=manifest["base_revision"])
    encoder = QwenAdapterEncoder(
        base, manifest["query_instruction"], max_seq_length, adapter_path=path
    )
    return encoder, adapter_identity(path, max_seq_length)

In [ ]:
"""Authenticated, bounded E5 inference; usable on a permitted notebook or GPU host."""

import asyncio
import hashlib
import json
import secrets
import threading
from pathlib import Path
from typing import Literal

from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, ConfigDict, ValidationError


def checkpoint_identity(path, max_seq_length=512, family="e5"):
    """Identity follows weights/tokenizer/config, never the temporary tunnel URL."""
    root = Path(path)
    files = sorted(
        p
        for p in root.rglob("*")
        if p.is_file()
        and p.suffix
        in {
            ".safetensors",
            ".bin",
            ".json",
            ".model",
            ".txt",
        }
        and "checkpoints" not in p.relative_to(root).parts
    )
    if not any(p.suffix in {".safetensors", ".bin"} for p in files):
        raise ValueError("Checkpoint must contain model weights and tokenizer/config files.")
    if family not in {"e5", "qwen3"}:
        raise ValueError("Unsupported embedding family.")
    digest = hashlib.sha256(f"{family}:query:passage:normalized:v1:{max_seq_length}".encode())
    for file in files:
        digest.update(str(file.relative_to(root)).replace("\\", "/").encode())
        with file.open("rb") as stream:
            while block := stream.read(1024 * 1024):
                digest.update(block)
    return f"research-{family}-" + digest.hexdigest()[:32]


class EmbeddingInput(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    model: str
    input: str | list[str]
    input_type: Literal["query", "passage"]


def create_embedding_app(
    encoder, api_key, model_id, max_seq_length=512, family="e5", tokenizer_model=""
):
    if len(api_key) < 32:
        raise ValueError("Use an API key with at least 32 characters.")
    if family not in {"e5", "qwen3"} or not model_id.startswith(f"research-{family}-"):
        raise ValueError("Model ID must match the embedding family.")
    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
    lock = threading.Lock()
    active = 0
    dimension = int(encoder.get_sentence_embedding_dimension())

    def authorize(request):
        header = request.headers.get("authorization", "")
        if not secrets.compare_digest(header, "Bearer " + api_key):
            raise HTTPException(401, "Invalid API key.")

    @app.get("/health")
    async def health():
        return {"status": "ok"}

    @app.get("/v1/info")
    async def info(request: Request):
        authorize(request)
        return {
            "model": model_id,
            "family": family,
            "tokenizer_model": tokenizer_model,
            "dimension": dimension,
            "max_seq_length": max_seq_length,
            "normalized": True,
            "protocol": "research-embedding-v1",
            "max_batch_size": 32,
        }

    def encode(texts, input_type):
        with lock:
            if family == "e5":
                method = encoder.encode
                inputs = [("query: " if input_type == "query" else "passage: ") + t for t in texts]
            else:
                method = encoder.encode_query if input_type == "query" else encoder.encode_document
                inputs = texts
            vectors = method(
                inputs,
                batch_size=8 if family == "e5" else 2,
                normalize_embeddings=True,
                show_progress_bar=False,
                convert_to_numpy=True,
            )
            return vectors.tolist()

    @app.post("/v1/embeddings")
    async def embeddings(request: Request):
        nonlocal active
        authorize(request)
        if active >= 4:
            raise HTTPException(429, "GPU queue is full; retry later.")
        body = bytearray()
        async for part in request.stream():
            body.extend(part)
            if len(body) > 300000:
                raise HTTPException(413, "Request is too large.")
        try:
            value = EmbeddingInput.model_validate_json(body)
        except (ValidationError, ValueError):
            raise HTTPException(422, "Invalid embedding request.") from None
        if value.model != model_id:
            raise HTTPException(409, "Checkpoint changed; configure the new model ID and reindex.")
        texts = [value.input] if isinstance(value.input, str) else value.input
        if not 1 <= len(texts) <= 32 or any(
            not text.strip() or len(text) > 16000 for text in texts
        ):
            raise HTTPException(422, "Use 1-32 nonempty texts, at most 16000 characters each.")
        if active >= 4:
            raise HTTPException(429, "GPU queue is full; retry later.")
        # Retain the queue slot and physical lock until the thread actually exits.
        active += 1
        job = asyncio.create_task(asyncio.to_thread(encode, texts, value.input_type))

        def finished(task):
            nonlocal active
            active -= 1
            if not task.cancelled():
                task.exception()  # Retrieve a late exception after an HTTP cancellation.

        job.add_done_callback(finished)
        try:
            vectors = await asyncio.shield(job)
        except asyncio.CancelledError:
            raise
        except Exception:
            raise HTTPException(
                503, "GPU inference failed; inspect the private server log."
            ) from None
        return {
            "object": "list",
            "model": model_id,
            "data": [
                {"object": "embedding", "index": i, "embedding": vector}
                for i, vector in enumerate(vectors)
            ],
        }

    return app


def extract_checkpoint(archive, output):
    """Extract a downloaded training checkpoint; reject traversal and symlinks."""
    import zipfile

    root = Path(output).resolve()
    if root.exists() and any(root.iterdir()):
        raise ValueError("Choose an empty checkpoint extraction directory.")
    with zipfile.ZipFile(archive) as zipped:
        total = 0
        for item in zipped.infolist():
            target = (root / item.filename.replace("\\", "/")).resolve()
            if not target.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Unsafe checkpoint archive entry.")
            total += item.file_size
            if total > 16 * 1024**3:
                raise ValueError("Checkpoint archive exceeds 16 GiB.")
        zipped.extractall(root)
    candidates = list(root.rglob("modules.json"))
    if len(candidates) != 1:
        raise ValueError("Use the checkpoint ZIP, not the full experiment/checkpoints ZIP.")
    return candidates[0].parent


def load_e5_checkpoint(path, device="cuda", max_seq_length=512):
    from sentence_transformers import SentenceTransformer

    root = Path(path)
    manifest = json.loads((root / "research_model.json").read_text(encoding="utf-8"))
    if manifest.get("model_family") != "e5" or not manifest.get("fine_tuned"):
        raise ValueError("Select an E5 checkpoint with actual optimizer steps (fine_tuned=true).")
    model = SentenceTransformer(
        str(root), device=device, local_files_only=True, trust_remote_code=False
    )
    model.max_seq_length = max_seq_length
    return model, checkpoint_identity(root, max_seq_length)


def load_qwen_checkpoint(path, device="cuda", max_seq_length=512):
    """Load a local Qwen3 embedding snapshot; never substitute a base model silently."""
    import torch
    from sentence_transformers import SentenceTransformer

    root = Path(path)
    if not root.is_dir():
        raise ValueError("Select a local Qwen3-Embedding checkpoint directory.")
    manifest_path = root / "research_model.json"
    if (
        manifest_path.is_file()
        and json.loads(manifest_path.read_text("utf-8")).get("format") == "swift_qwen3_adapter_v1"
    ):
        loader = globals().get("load_swift_qwen_adapter")
        if loader is None:
            from notebooks.qwen_adapter_encoder import load_swift_qwen_adapter

            loader = load_swift_qwen_adapter
        return loader(root, max_seq_length)
    config_files = list(root.glob("config.json")) or list(root.glob("0_Transformer/config.json"))
    if (
        not config_files
        or json.loads(config_files[0].read_text(encoding="utf-8")).get("model_type") != "qwen3"
    ):
        raise ValueError("Checkpoint is not a Qwen3 embedding model.")
    model = SentenceTransformer(
        str(root),
        device=device,
        local_files_only=True,
        trust_remote_code=False,
        model_kwargs={"torch_dtype": torch.float16},
    )
    model.max_seq_length = max_seq_length
    if not hasattr(model, "encode_query") or not hasattr(model, "encode_document"):
        raise ValueError("Checkpoint does not support query/document embedding.")
    return model, checkpoint_identity(root, max_seq_length, family="qwen3")

In [ ]:
# Select one model, or set LOCAL_CHECKPOINT to a Qwen3 Embedding directory.
BASE_MODEL = "Qwen/Qwen3-Embedding-4B"  # or Qwen/Qwen3-Embedding-0.6B
LOCAL_CHECKPOINT = ""  # extracted qwen3-adapter folder from notebook 06, or full checkpoint
ADAPTER_ZIP = ""  # e.g. /kaggle/input/my-training/qwen3-adapter.zip from notebook 06
MAX_SEQ_LENGTH = 512  # start conservatively on T4
PORT = 8008
START_GPU_SERVER = False
START_CLOUDFLARE = False
API_KEY_SECRET_NAME = "RESEARCH_GPU_API_KEY"

In [ ]:
if START_GPU_SERVER:
    import gc, getpass, threading, time
    import httpx, torch, uvicorn
    from pathlib import Path
    from huggingface_hub import snapshot_download
    if "GPU_SERVER" in globals():
        raise RuntimeError("Stop the previous server before starting another one.")
    if BASE_MODEL not in {"Qwen/Qwen3-Embedding-0.6B", "Qwen/Qwen3-Embedding-4B"}:
        raise ValueError("Choose a supported Qwen3 Embedding base model.")
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator first.")
    if ADAPTER_ZIP:
        selected = extract_adapter_archive(ADAPTER_ZIP, "/kaggle/temp/qwen-adapter-input")
    else:
        selected = (Path(LOCAL_CHECKPOINT) if LOCAL_CHECKPOINT
                    else Path(snapshot_download(BASE_MODEL)))
    manifest_path = selected / "research_model.json"
    if (manifest_path.is_file()
            and json.loads(manifest_path.read_text()).get("format") == "swift_qwen3_adapter_v1"):
        adapter_info = read_adapter_manifest(selected)
        BASE_MODEL = adapter_info["base_model"]
        MAX_SEQ_LENGTH = adapter_info["max_seq_length"]
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret(API_KEY_SECRET_NAME)
    except Exception:
        API_KEY = getpass.getpass("Private GPU API key (at least 32 characters): ")
    if len(API_KEY) < 32:
        raise ValueError("Use a key with at least 32 characters.")
    GPU_ENCODER, GPU_MODEL_ID = load_qwen_checkpoint(selected, "cuda", MAX_SEQ_LENGTH)
    GPU_APP = create_embedding_app(GPU_ENCODER, API_KEY, GPU_MODEL_ID,
                                   MAX_SEQ_LENGTH, family="qwen3", tokenizer_model=BASE_MODEL)
    GPU_SERVER = uvicorn.Server(uvicorn.Config(GPU_APP, host="127.0.0.1", port=PORT,
                               log_level="warning", access_log=False))
    GPU_THREAD = threading.Thread(target=GPU_SERVER.run, daemon=True)
    GPU_THREAD.start()
    deadline = time.monotonic() + 60
    while not GPU_SERVER.started and GPU_THREAD.is_alive() and time.monotonic() < deadline:
        time.sleep(0.2)
    if not GPU_SERVER.started:
        raise RuntimeError("GPU server did not start; inspect private notebook log.")
    check = httpx.post(f"http://127.0.0.1:{PORT}/v1/embeddings",
        headers={"Authorization": "Bearer " + API_KEY},
        json={"model": GPU_MODEL_ID, "input": ["Find relevant research evidence"],
              "input_type": "query"}, timeout=120)
    check.raise_for_status()
    print("GPU:", torch.cuda.get_device_name(0))
    print("Model ID:", GPU_MODEL_ID)
    print("Dimension:", len(check.json()["data"][0]["embedding"]))
    print("Local API test passed. API key is not printed.")
else:
    print("Server disabled. Set START_GPU_SERVER=True when ready.")

In [ ]:
if START_CLOUDFLARE:
    import hashlib, json, re, shutil, subprocess, threading, time, urllib.request
    if "GPU_SERVER" not in globals() or not GPU_SERVER.started:
        raise RuntimeError("Start the GPU server and pass its local API check before tunneling.")
    if "CLOUDFLARE_PROCESS" in globals() and CLOUDFLARE_PROCESS.poll() is None:
        raise RuntimeError("Stop the existing tunnel before creating another one.")
    binary = shutil.which("cloudflared")
    if not binary:
        # Resolve an official release and verify GitHub's published asset digest.
        req = urllib.request.Request("https://api.github.com/repos/cloudflare/cloudflared/releases/latest",
                                     headers={"User-Agent": "Research-assist-notebook"})
        with urllib.request.urlopen(req, timeout=30) as response:
            release = json.load(response)
        asset = next(a for a in release["assets"] if a["name"] == "cloudflared-linux-amd64")
        digest = asset.get("digest", "")
        if not digest.startswith("sha256:"):
            raise RuntimeError("No published SHA256 digest. Install verified cloudflared manually.")
        binary = "/kaggle/temp/cloudflared"
        urllib.request.urlretrieve(asset["browser_download_url"], binary)
        with open(binary, "rb") as stream:
            actual = hashlib.file_digest(stream, "sha256").hexdigest()
        if actual != digest.split(":", 1)[1]:
            raise RuntimeError("cloudflared checksum mismatch; refusing execution.")
        os.chmod(binary, 0o700)
    CLOUDFLARE_PROCESS = subprocess.Popen([binary, "tunnel", "--no-autoupdate", "--url",
        f"http://127.0.0.1:{PORT}"], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    TUNNEL_URL = None
    TUNNEL_READY = threading.Event()
    def read_tunnel_log():
        global TUNNEL_URL
        for line in CLOUDFLARE_PROCESS.stderr:
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if match:
                TUNNEL_URL = match.group(0)
                TUNNEL_READY.set()
    threading.Thread(target=read_tunnel_log, daemon=True).start()
    if not TUNNEL_READY.wait(60):
        CLOUDFLARE_PROCESS.terminate()
        raise RuntimeError("No tunnel URL received. Check network/platform permissions.")
    print("Copy this URL:", TUNNEL_URL)
    print("\nLocal app .env (replace existing entries):")
    print("RA_EMBEDDING_PROVIDER=remote")
    print("RA_EMBEDDING_BASE_URL=" + TUNNEL_URL + "/v1")
    print("RA_EMBEDDING_MODEL=" + GPU_MODEL_ID)
    print("RA_EMBEDDING_API_KEY=<the same private key you entered above>")
    print("RA_EMBEDDING_MAX_SEQ_LENGTH=" + str(MAX_SEQ_LENGTH))
    print("RA_EMBEDDING_BATCH_SIZE=16")
    print("RA_CHUNK_STRATEGY=tokens")
    print("RA_CHUNK_TOKENS=384")
    print("RA_CHUNK_TOKEN_OVERLAP=48")
    print("RA_CHUNK_TOKENIZER_MODEL=" + BASE_MODEL)
    print("Restart the app. A new checkpoint requires Reindex.")
    print("A new URL for the same model ID does not require Reindex.")
else:
    print("Tunnel disabled. Set START_CLOUDFLARE=True only if your host permits it.")

In [ ]:
# Run explicitly when you finish. This cell is not part of Run All.
STOP_ENDPOINT = False
if STOP_ENDPOINT:
    if "CLOUDFLARE_PROCESS" in globals() and CLOUDFLARE_PROCESS.poll() is None:
        CLOUDFLARE_PROCESS.terminate()
        CLOUDFLARE_PROCESS.wait(timeout=15)
    if "GPU_SERVER" in globals():
        GPU_SERVER.should_exit = True
        GPU_THREAD.join(timeout=60)
        if GPU_THREAD.is_alive():
            raise RuntimeError("GPU work is still draining. Wait before unloading the model.")
        del GPU_SERVER, GPU_APP, GPU_ENCODER
        gc.collect()
        torch.cuda.empty_cache()
    print("Endpoint stopped.")